# Use Case 4 — Four-Layer T-Logic for EPC Compliance

**Goal:** Implement a four-layer Temporal Logic (T-Logic) framework for rule-based compliance prediction on the EPC TKG.

## Architecture: Four Layers
| Layer | Content | Role |
|-------|---------|------|
| **L1 Entity** | Workers, Steps, Certs, Permits | Graph nodes |
| **L2 Relation** | HAS_CERT, ASSIGNED_TO, REQUIRES_PERMIT, PERMIT_DENIED | Edge types |
| **L3 Temporal** | valid_from/valid_to, tx_time, rule_change @ month 6 | Bitemporal windows |
| **L4 Rule** | Mined T-Logic rules: body atoms -> head prediction | Compliance inference |

**T-Logic** mines cyclic temporal rules of the form:
```
r_h(X, Y, t) <- r_1(X, Z_1, t+d_1) AND r_2(Z_1, Z_2, t+d_2) AND ... AND r_n(Z_{n-1}, Y, t+d_n)
```

Applied here to predict `PERMIT_DENIED` violations from cert and assignment patterns.

**Prior notebooks:** 05: TGN event stream | 06: TNTComplEx + TGN cert-aware | **07 (this):** Symbolic T-Logic


## 0. Setup


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime, timedelta, timezone
from collections import defaultdict, Counter
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../../data/UseCase4')
RULE_CHANGE   = datetime(2024, 6, 29, tzinfo=timezone.utc)
PROJECT_START = datetime(2024, 1, 1,  tzinfo=timezone.utc)
PROJECT_END   = datetime(2025, 7, 1,  tzinfo=timezone.utc)
print('Setup OK')


## 1. Load Data


In [ ]:
with open(DATA_DIR / 'epc_dataset_real.json', encoding='utf-8') as f:
    dataset = json.load(f)
with open(DATA_DIR / 'epc_events.json', encoding='utf-8') as f:
    events = json.load(f)

steps    = dataset['steps']
workers  = dataset['workers']
certs    = dataset['certifications']
seqs     = dataset['step_sequences']
update   = dataset['update_events'][0]
assigned = events['assigned_to']
denied   = events['permit_denied']

m = dataset['metadata']
print(f'Steps: {m["total_steps"]} | Workers: {m["total_workers"]} | Permits: {m["total_permits"]} | Certs: {m["total_certs"]}')
print(f'Events -> assigned: {len(assigned)} | permit_denied: {len(denied)}')
print(f'Rule change: {update["valid_from"]} -> +{update["new_cert"]}')


## 2. Layer 1 — Entity Index


In [ ]:
worker_map = {w['id']: w for w in workers}
step_map   = {s['id']: s for s in steps}

def build_cert_index(workers):
    idx = {}
    for w in workers:
        idx[w['id']] = {}
        for c in w['certifications']:
            cid = c['cert'].replace(' ', '_')
            idx[w['id']][cid] = (
                datetime.fromisoformat(c['valid_from']),
                datetime.fromisoformat(c['valid_to'])
            )
    return idx

cert_idx = build_cert_index(workers)
print(f'Workers: {len(worker_map)} | Steps: {len(step_map)} | Cert index: {len(cert_idx)} workers')


## 3. Layer 2 — Relation Quadruples
Build `(subject, relation, object, t_start, t_end)` for all edge types.


In [ ]:
quads = []

for w in workers:
    for c in w['certifications']:
        quads.append((w['id'], 'HAS_CERT', c['cert'].replace(' ', '_'),
                      datetime.fromisoformat(c['valid_from']),
                      datetime.fromisoformat(c['valid_to'])))

for s in steps:
    quads.append((s['id'], 'REQUIRES_PERMIT', s['permit_type'],
                  datetime.fromisoformat(s['valid_from']),
                  datetime.fromisoformat(s['valid_to'])))

for e in assigned:
    t = datetime.fromisoformat(e['date'])
    quads.append((e['worker_id'], 'ASSIGNED_TO', e['step_id'], t, t))

for e in denied:
    t = datetime.fromisoformat(e['date'])
    quads.append((e['worker_id'], 'PERMIT_DENIED', e['step_id'], t, t))

for seq in seqs:
    ta = datetime.fromisoformat(step_map[seq['from']]['valid_from'])
    tb = datetime.fromisoformat(step_map[seq['to']]['valid_from'])
    quads.append((seq['from'], 'PRECEDES', seq['to'], ta, tb))

df_quads = pd.DataFrame(quads, columns=['subject','relation','object','t_start','t_end'])
print(f'Total quadruples: {len(df_quads)}')
print(df_quads.groupby('relation').size().to_string())


## 4. Layer 3 — Temporal Windows
Partition around the rule-change event at month 6.


In [ ]:
windows = [
    ('PRE_RULE',  PROJECT_START, RULE_CHANGE),
    ('POST_RULE', RULE_CHANGE,   PROJECT_END),
    ('FULL',      PROJECT_START, PROJECT_END),
]

print('Quadruples per temporal window:')
for name, ts, te in windows:
    sub = df_quads[(df_quads['t_start'] >= ts) & (df_quads['t_start'] < te)]
    d_w = sub[sub['relation'] == 'PERMIT_DENIED']
    a_w = sub[sub['relation'] == 'ASSIGNED_TO']
    rate = len(d_w)/len(a_w)*100 if len(a_w) > 0 else 0
    print(f'  {name:<12}: {len(sub):>5} quads | {len(d_w):>3} denied / {len(a_w):>4} assigned ({rate:.1f}%)')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.hist([datetime.fromisoformat(e['date']).timestamp() for e in assigned],
        bins=30, alpha=0.6, label='ASSIGNED_TO', color='steelblue')
ax.hist([datetime.fromisoformat(e['date']).timestamp() for e in denied],
        bins=30, alpha=0.9, label='PERMIT_DENIED', color='crimson')
ax.axvline(RULE_CHANGE.timestamp(), color='orange', linestyle='--', lw=2, label='Rule Change')
xticks = pd.date_range('2024-01-01', '2025-07-01', freq='3ME')
ax.set_xticks([x.timestamp() for x in xticks])
ax.set_xticklabels([x.strftime('%b %Y') for x in xticks], rotation=45)
ax.set_title('Event Timeline')
ax.set_ylabel('Count')
ax.legend()

ax2 = axes[1]
permit_counts = Counter(e['permit_type'] for e in assigned)
denied_counts  = Counter(e['permit_type'] for e in denied)
pts = sorted(permit_counts.keys())
x = np.arange(len(pts))
w = 0.35
ax2.bar(x-w/2, [permit_counts[p] for p in pts], w, label='Assigned', color='steelblue', alpha=0.7)
ax2.bar(x+w/2, [denied_counts.get(p,0) for p in pts], w, label='Denied', color='crimson', alpha=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels(pts, rotation=40, ha='right')
ax2.set_title('Assignments vs Violations by Permit Type')
ax2.legend()

plt.tight_layout()
plt.show()
print(f'PERMIT_DENIED: {len(denied)} ({len(denied)/len(assigned)*100:.1f}% of assignments)')


## 5. Layer 4 — T-Logic Rule Mining

Three rule templates:
- **R1:** `ASSIGNED_TO(W,S,t) AND missing_cert(W, permit_type(S), t)` -> `PERMIT_DENIED`
- **R2:** `ASSIGNED_TO(W,S,t) AND t>=RULE_CHANGE AND permit=hot_work AND NOT HAS_CERT(W, Advanced_Fire_Watch, t)` -> `PERMIT_DENIED`
- **R3:** `PERMIT_DENIED(W,S1,t) AND PRECEDES(S1,S2)` -> `cascade_risk(S2, t+delta)`


In [ ]:
CERT_REQS = {
    'hot_work':       {'Hot_Work_Safety', 'Fire_Watch', 'Welding_Certification'},
    'excavation':     {'Excavation_Safety', 'Confined_Space_Entry', 'Soil_Assessment'},
    'lifting':        {'Rigging_&_Lifting', 'Crane_Operator', 'Slinging_Certificate'},
    'electrical':     {'Electrical_Safety', 'LOTO_Certification', 'HV_Awareness'},
    'confined_space': {'Confined_Space_Entry', 'Gas_Testing', 'Emergency_Response'},
    'radiography':    {'NDT_Level_II', 'Radiation_Safety', 'RT_Operator'},
    'work_at_height': {'Working_at_Height', 'Scaffold_Inspection', 'Fall_Arrest'},
    'general_work':   set(),
}
HOT_WORK_NEW = 'Advanced_Fire_Watch'

def cert_valid_at(wid, cid, t):
    entry = cert_idx.get(wid, {}).get(cid)
    if entry is None: return False
    vf, vt = entry
    return vf <= t <= vt

def worker_qualifies(wid, permit, t):
    if permit == 'general_work': return True, []
    req = set(CERT_REQS.get(permit, set()))
    if t >= RULE_CHANGE and permit == 'hot_work':
        req.add(HOT_WORK_NEW)
    missing = [c for c in req if not cert_valid_at(wid, c, t)]
    return len(missing) == 0, missing

# Sanity check
t_pre  = datetime(2024, 3, 1, tzinfo=timezone.utc)
t_post = datetime(2024, 8, 1, tzinfo=timezone.utc)
print('W-001 hot_work pre-rule: ', worker_qualifies('W-001', 'hot_work', t_pre))
print('W-001 hot_work post-rule:', worker_qualifies('W-001', 'hot_work', t_post))


In [ ]:
denied_set = {(e['worker_id'], e['step_id']) for e in denied}

records = []
for e in assigned:
    wid    = e['worker_id']
    sid    = e['step_id']
    t      = datetime.fromisoformat(e['date'])
    permit = e['permit_type']
    after_rc = t >= RULE_CHANGE
    _, missing = worker_qualifies(wid, permit, t)
    records.append({
        'worker': wid, 'step': sid, 'permit': permit,
        't': t, 'after_rc': after_rc,
        'missing_cert': len(missing) > 0,
        'n_missing': len(missing),
        'is_denied': (wid, sid) in denied_set,
    })

df_m = pd.DataFrame(records)
print(f'Rule instances: {len(df_m)}')
print(f'  missing cert: {df_m["missing_cert"].sum()}')
print(f'  is_denied:    {df_m["is_denied"].sum()}')


### 5.1 Rule Confidence & Metrics


In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

y_true = df_m['is_denied'].astype(int).values
y_r1   = df_m['missing_cert'].astype(int).values
y_r2   = ((df_m['permit']=='hot_work') & df_m['after_rc'] & df_m['missing_cert']).astype(int).values
y_comb = np.clip(y_r1 + y_r2, 0, 1)

print(f'{'Rule':<22} {'Support':>8} {'Conf':>6} {'P':>6} {'R':>6} {'F1':>6}')
print('-'*58)
for name, y_pred in [('R1 MissingCert', y_r1), ('R2 PostRC_HotWork', y_r2), ('R1+R2 Combined', y_comb)]:
    sup  = int(y_pred.sum())
    conf = round(int((y_true & y_pred).sum()) / max(sup,1), 3)
    p    = round(precision_score(y_true, y_pred, zero_division=0), 3)
    r    = round(recall_score(y_true, y_pred, zero_division=0), 3)
    f    = round(f1_score(y_true, y_pred, zero_division=0), 3)
    print(f'{name:<22} {sup:>8} {conf:>6} {p:>6} {r:>6} {f:>6}')


### 5.2 Temporal Train/Test: Pre vs Post Rule Change


In [ ]:
df_train = df_m[~df_m['after_rc']]
df_test  = df_m[ df_m['after_rc']]
print(f'Train (pre):  {len(df_train):>4} rows | {df_train["is_denied"].sum()} violations')
print(f'Test  (post): {len(df_test):>4} rows | {df_test["is_denied"].sum()} violations')

y_te   = df_test['is_denied'].astype(int).values
y_r1t  = df_test['missing_cert'].astype(int).values
y_r2t  = ((df_test['permit']=='hot_work') & df_test['after_rc'] & df_test['missing_cert']).astype(int).values
y_ct   = np.clip(y_r1t + y_r2t, 0, 1)

print('\n--- R1+R2 on TEST set (post rule-change) ---')
print(classification_report(y_te, y_ct, target_names=['OK','DENIED'], zero_division=0))


In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, y_pred, title in [(axes[0], y_r1t, 'R1: Missing Cert (test)'),
                           (axes[1], y_ct,  'R1+R2 Combined (test)')]:
    cm = confusion_matrix(y_te, y_pred)
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Pred OK','Pred DENIED'])
    ax.set_yticklabels(['True OK','True DENIED'])
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
plt.tight_layout()
plt.show()


## 6. Temporal Walk (T-Logic Path Scoring)


In [ ]:
adj = defaultdict(lambda: defaultdict(list))
for _, row in df_quads.iterrows():
    adj[row['subject']][row['relation']].append((row['object'], row['t_start']))

def temporal_walk(start, relations):
    frontier = [(start, datetime.min.replace(tzinfo=timezone.utc), [start])]
    for rel in relations:
        nxt = []
        for node, t_cur, path in frontier:
            for obj, t_obj in adj[node][rel]:
                if t_obj >= t_cur:
                    nxt.append((obj, t_obj, path + [obj]))
        frontier = nxt
    return frontier

walks = temporal_walk('W-001', ['ASSIGNED_TO', 'REQUIRES_PERMIT'])
print(f'W-001 ASSIGNED_TO -> REQUIRES_PERMIT paths: {len(walks)}')
for obj, t, path in walks[:5]:
    print(f'  {"/".join(path)} @ {t.date()}')


In [ ]:
rule_stats = []
for permit in CERT_REQS:
    sub  = df_m[df_m['permit'] == permit]
    body = int(sub['missing_cert'].sum())
    tp   = int((sub['missing_cert'] & sub['is_denied']).sum())
    conf = tp / max(body, 1)
    rule_stats.append({'Permit': permit, 'Body fires': body,
                       'TP': tp, 'Confidence': round(conf, 3), 'Total': len(sub)})

df_rs = pd.DataFrame(rule_stats).sort_values('Confidence', ascending=False)
print('T-Logic Rule Confidence by Permit Type:')
print(df_rs.to_string(index=False))


## 7. Rule 3 — Cascade Risk (Delay Propagation)


In [ ]:
successors = defaultdict(list)
for seq in seqs:
    successors[seq['from']].append(seq['to'])

def cascade_risk(denied_steps, max_depth=3):
    risk     = defaultdict(float)
    frontier = {sid: 1.0 for sid in denied_steps}
    visited  = set(denied_steps)
    for depth in range(max_depth):
        decay = 0.5 ** (depth + 1)
        nxt   = {}
        for sid, base in frontier.items():
            for succ in successors.get(sid, []):
                if succ not in visited:
                    nxt[succ]   = max(nxt.get(succ, 0), base * decay)
                    risk[succ] += base * decay
                    visited.add(succ)
        frontier = nxt
    return dict(risk)

denied_sids = [e['step_id'] for e in denied]
cascade = cascade_risk(denied_sids)

df_cascade = pd.DataFrame([
    {'step_id': sid, 'risk': score,
     'discipline': step_map.get(sid, {}).get('discipline', '?'),
     'permit': step_map.get(sid, {}).get('permit_type', '?')}
    for sid, score in sorted(cascade.items(), key=lambda x: -x[1])
])
print(f'Steps at cascade risk: {len(df_cascade)}')
print(df_cascade.head(10).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

risk_by_disc = df_cascade.groupby('discipline')['risk'].sum().sort_values(ascending=False)
risk_by_disc.plot.bar(ax=axes[0], color='darkorange', alpha=0.8)
axes[0].set_title('Cascade Delay Risk by Discipline')
axes[0].set_ylabel('Total Risk Score')
axes[0].tick_params(axis='x', rotation=30)

top15 = df_cascade.head(15)
axes[1].barh(top15['step_id'], top15['risk'], color='darkorange', alpha=0.8)
axes[1].set_title('Top 15 Steps at Cascade Risk')
axes[1].set_xlabel('Risk Score')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


## 8. Comparison: T-Logic vs Embedding Models


In [ ]:
p_r1 = precision_score(y_true, y_r1,   zero_division=0)
r_r1 = recall_score(y_true,    y_r1,   zero_division=0)
f_r1 = f1_score(y_true,        y_r1,   zero_division=0)
p_cb = precision_score(y_true, y_comb, zero_division=0)
r_cb = recall_score(y_true,    y_comb, zero_division=0)
f_cb = f1_score(y_true,        y_comb, zero_division=0)

comparison = pd.DataFrame([
    {'Model': 'TNTComplEx (Nb06)',       'P': 0.62, 'R': 0.58, 'F1': 0.60, 'Type': 'Embedding'},
    {'Model': 'TGN Cert-Aware (Nb06)',   'P': 0.71, 'R': 0.65, 'F1': 0.68, 'Type': 'Embedding'},
    {'Model': 'T-Logic R1 (Nb07)',       'P': round(p_r1,2), 'R': round(r_r1,2), 'F1': round(f_r1,2), 'Type': 'Symbolic'},
    {'Model': 'T-Logic R1+R2 (Nb07)',    'P': round(p_cb,2), 'R': round(r_cb,2), 'F1': round(f_cb,2), 'Type': 'Symbolic'},
])
print(comparison.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
models = comparison['Model'].tolist()
x = np.arange(len(models))
w = 0.25
tc = {'Embedding': 'steelblue', 'Symbolic': 'darkorange'}
colors = [tc[t] for t in comparison['Type']]

for metric, offset, alpha in [('P', -w, 0.6), ('R', 0, 0.8), ('F1', w, 1.0)]:
    vals = comparison[metric].tolist()
    bars = ax.bar(x + offset, vals, w, color=colors, alpha=alpha, label=metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.2f}',
                ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=20, ha='right')
ax.set_ylim(0, 1.15)
ax.set_title('Model Comparison: Embedding vs Symbolic T-Logic')
ax.set_ylabel('Score')
patch_e = mpatches.Patch(color='steelblue',  label='Embedding')
patch_s = mpatches.Patch(color='darkorange', label='Symbolic T-Logic')
ax.legend(handles=[patch_e, patch_s])
plt.tight_layout()
plt.show()


## 9. Summary

| Layer | Implementation | Key Insight |
|-------|---------------|-------------|
| **L1 Entity** | 50 workers, 1518 steps, 33 certs, 8 permits | Real TR Family Steps data |
| **L2 Relation** | ASSIGNED_TO, HAS_CERT, REQUIRES_PERMIT, PERMIT_DENIED, PRECEDES | 5 relation types |
| **L3 Temporal** | Bitemporal windows, rule change at month 6 | Pre/post split reveals rule-change effect |
| **L4 Rule** | R1: missing cert; R2: post-RC hot_work; R3: cascade | Fully interpretable |

**Key Findings:**
- T-Logic symbolic rules achieve competitive F1 vs embedding models with full interpretability
- R2 captures the bitemporal violation pattern exactly — impossible without temporal structure
- Cascade analysis (R3) identifies downstream steps at risk from upstream permit denials
- Symbolic rules are explainable by design — critical for EPC compliance audit

**Next steps:** Hybrid model (T-Logic constraints in TGN training) | Apply to real TR data | Export rules to Neo4j APOC
